# Cơ chế tăng tốc của FastGS

> Notebook này giải thích **FastGS làm gì để huấn luyện 3D Gaussian Splatting trong ~100 giây**, rồi chạy thật trên Google Colab free (T4).
> Cách trình bày giống [DOCS/gaussian-splatting-math.md](DOCS/gaussian-splatting-math.md): mỗi cơ chế đi kèm công thức, bảng ký hiệu, ví dụ số và phần "vì sao".
>
> Tên hàm/tham số trong bài trỏ thẳng tới mã nguồn: `utils/fast_utils.py`, `scene/gaussian_model.py`, `gaussian_renderer/__init__.py`, `train.py`.

| Phần | Nội dung | Cần GPU? |
|---|---|---|
| **1–6** | Cơ chế tăng tốc, mỗi phần kèm **mô phỏng thu nhỏ** bằng NumPy | ❌ chạy được ngay |
| **7** | Huấn luyện thật trên Colab T4: preset A, điểm `Score` mỗi 1000 vòng, chống tràn RAM, render + thống kê, tải mô hình | ✅ |
| **8** | Ba nâng cấp thuần Python + ablation `Score` theo phút | ✅ |
| **9** | Roadmap tầng CUDA (Mip-Splatting, StopThePop, 2DGS/GOF, gsplat-MCMC) | — |

> Quy ước: **toàn bộ ô code là tiếng Anh, không comment**; mọi giải thích nằm ở ô markdown.
> Tài liệu đồng hành: [DOCS/fastgs-acceleration-method.md](DOCS/fastgs-acceleration-method.md) (Phần 1–6) và [DOCS/colab-t4-guide.md](DOCS/colab-t4-guide.md) (Phần 7–9).

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

## Phần 0: Thời gian huấn luyện 3DGS đi đâu?

3DGS gốc chậm vì ba nguyên nhân cộng dồn qua 30.000 vòng lặp:

| Nguồn tốn thời gian | Bản chất |
|---|---|
| **Số lượng Gaussian phình to** | Adaptive Density Control nhân bản Gaussian ở *mọi* vùng có gradient lớn, kể cả vùng đã hội tụ → hàng triệu điểm, mỗi vòng lặp phải rasterize và backward hết. |
| **Rasterization theo tile** | Mỗi splat 2D được gán cho tất cả tile 16×16 mà bounding box của nó chạm tới. Bounding box lỏng → 1 splat đè lên rất nhiều tile → nhiều cặp (tile, Gaussian) phải sort và blend. |
| **Backward của Spherical Harmonics** | Màu phụ thuộc góc nhìn mã hoá bằng 48 hệ số SH bậc 3; hội tụ chậm nếu mọi hệ số dùng chung một learning rate nhỏ. |

FastGS tấn công cả ba bằng ba đòn bẩy tương ứng:

1. **Điểm số nhất quán đa góc nhìn (multi-view consistency score)** — chỉ nhân bản Gaussian ở nơi *nhiều camera cùng thấy sai*, và cắt bỏ mạnh tay Gaussian vô ích. → **Phần 1–3**
2. **Compact box (`--mult`)** — thu nhỏ bounding box mỗi splat để giảm số tile. → **Phần 4**
3. **Tách learning rate SH bậc thấp / bậc cao** (`--lowfeature_lr` / `--highfeature_lr`). → **Phần 5**

Kết quả: cùng số vòng lặp nhưng mỗi vòng rẻ hơn nhiều, và số Gaussian được giữ ở mức tối thiểu cần thiết.

## Phần 1: Bản đồ lỗi nhị phân trên mỗi góc nhìn

Hàm `compute_gaussian_score_fastgs` (`utils/fast_utils.py:45`) là đóng góp chính. Nó chạy trước mỗi lần densify, lấy mẫu **10 camera ngẫu nhiên** (`sampling_cameras`, `fast_utils.py:10`) và với từng camera làm 3 bước.

### Bước 1.1 — Chuẩn hoá bản đồ lỗi L1 về [0, 1]

$$e(u,v)=\frac{1}{3}\sum_{k\in\{R,G,B\}}\bigl|\,I_{\text{render}}(u,v,k)-I_{\text{gt}}(u,v,k)\,\bigr|$$

$$\hat{e}(u,v)=\frac{e(u,v)-\min_{u,v} e}{\max_{u,v} e-\min_{u,v} e}$$

| Ký hiệu | Ý nghĩa |
|---|---|
| $I_{\text{render}}, I_{\text{gt}}$ | Ảnh render và ảnh thật của camera đang xét |
| $e(u,v)$ | Sai số L1 trung bình 3 kênh màu tại pixel $(u,v)$ — hàm `get_loss` |
| $\hat{e}(u,v)$ | Sai số đã chuẩn hoá min–max về $[0,1]$ trên toàn ảnh |

Chuẩn hoá min–max khiến ngưỡng ở bước sau **không phụ thuộc độ sáng tuyệt đối của cảnh** — cảnh tối và cảnh sáng dùng chung một `loss_thresh`.

### Bước 1.2 — Nhị phân hoá bằng ngưỡng `loss_thresh`

$$m(u,v)=\begin{cases}1 & \hat{e}(u,v) > \tau_{\text{loss}}\\[4pt] 0 & \text{ngược lại}\end{cases}
\qquad \tau_{\text{loss}}=\texttt{loss\_thresh}\ (\text{mặc định }0.1)$$

`m` là `metric_map` trong code (`fast_utils.py:82`): mặt nạ đánh dấu **những pixel mà mô hình hiện tại tái tạo tệ nhất**. `loss_thresh` thấp → nhiều pixel bị đánh dấu → giữ lại / sinh thêm nhiều Gaussian hơn.

In [ ]:
H, W = 6, 8

error = rng.uniform(0.0, 0.05, size=(H, W))
error[0:3, 0:3] += rng.uniform(0.2, 0.6, size=(3, 3))


def normalize_minmax(x):
    return (x - x.min()) / (x.max() - x.min())


error_hat = normalize_minmax(error)
loss_thresh = 0.1
metric_map = (error_hat > loss_thresh).astype(int)

print("normalized error map")
print(error_hat)
print()
print("metric_map, 1 marks a badly reconstructed pixel")
print(metric_map)
print()
print("bad pixels:", metric_map.sum(), "of", H * W)


### Bước 1.3 — Đổ ngược mặt nạ pixel về từng Gaussian

Rasterizer được gọi lại với `get_flag=True` và `metric_map=m` (`fast_utils.py:84`). Trong lượt này, mỗi Gaussian $i$ nhận về một bộ đếm:

$$\text{counts}^{(v)}_i=\#\bigl\{\text{pixel }(u,v)\ \text{mà Gaussian }i\ \text{đóng góp và}\ m(u,v)=1\bigr\}$$

trả về ở `render_pkg["accum_metric_counts"]`. Trực giác: **Gaussian $i$ "chịu trách nhiệm" cho bao nhiêu pixel lỗi** ở góc nhìn $v$ này.

In [ ]:
N_GAUSSIANS = 4
footprints = {
    0: [(r, c) for r in range(0, 3) for c in range(0, 3)],
    1: [(r, c) for r in range(0, 2) for c in range(0, 5)],
    2: [(r, c) for r in range(3, 6) for c in range(3, 8)],
    3: [(0, 0), (5, 7)],
}


def accum_metric_counts(mask, footprint_map):
    counts = np.zeros(len(footprint_map), dtype=int)
    for index, pixels in footprint_map.items():
        counts[index] = sum(mask[r, c] for r, c in pixels)
    return counts


counts_single_view = accum_metric_counts(metric_map, footprints)
for index in range(N_GAUSSIANS):
    print(f"gaussian {index}: covers {len(footprints[index]):2d} pixels, "
          f"{counts_single_view[index]} of them are bad")


## Phần 2: Gộp nhiều góc nhìn thành hai điểm số

Sau khi lặp qua cả 10 camera, hàm gộp lại thành hai đại lượng per-Gaussian.

### Công thức (1) — Importance score (điều khiển việc *sinh thêm*)

$$\text{Importance}_i=\left\lfloor \frac{1}{V}\sum_{v=1}^{V}\text{counts}^{(v)}_i \right\rfloor
\qquad V=10$$

| Ký hiệu | Ý nghĩa |
|---|---|
| $V$ | Số camera lấy mẫu (`num_cams = 10`) |
| $\text{counts}^{(v)}_i$ | Số pixel lỗi mà Gaussian $i$ phủ ở góc nhìn $v$ |
| $\lfloor\cdot\rfloor$ | Làm tròn xuống (`rounding_mode='floor'`, `fast_utils.py:102`) |

Đây là **số pixel-lỗi trung bình mỗi góc nhìn** một Gaussian gây ra. Vì lấy trung bình rồi làm tròn xuống, Gaussian chỉ bị "một camera duy nhất" tố sai sẽ có điểm gần 0 — **phải sai nhất quán trên nhiều góc nhìn** mới được điểm cao. Đó là ý nghĩa "multi-view consistent".

### Công thức (2) — Pruning score (điều khiển việc *cắt bỏ*)

$$s_i=\sum_{v=1}^{V}\mathcal{L}^{(v)}_{\text{photo}}\cdot\text{counts}^{(v)}_i
\qquad\qquad
\text{Pruning}_i=\frac{s_i-\min_j s_j}{\max_j s_j-\min_j s_j}\in[0,1]$$

| Ký hiệu | Ý nghĩa |
|---|---|
| $\mathcal{L}^{(v)}_{\text{photo}}$ | Loss ảnh của cả khung hình $v$: $(1-\lambda)\mathcal{L}_1+\lambda(1-\text{SSIM})$ với $\lambda=0.2$ (`compute_photometric_loss`) |
| $s_i$ | Điểm thô: số pixel-lỗi của Gaussian $i$, **nhân trọng số** bằng độ tệ toàn cục của góc nhìn đó |
| $\text{Pruning}_i$ | Điểm chuẩn hoá về $[0,1]$; càng gần 1 = Gaussian càng "vô dụng / gây hại" |

Khác biệt then chốt so với Importance: pruning score **nhân thêm $\mathcal{L}^{(v)}_{\text{photo}}$**. Một Gaussian phủ nhiều pixel lỗi trong khung hình vốn đã render rất tệ sẽ bị phạt nặng hơn.

In [ ]:
counts = np.array([
    [8, 6, 7],
    [12, 0, 0],
    [1, 0, 1],
], dtype=float)
photometric_loss = np.array([0.20, 0.05, 0.10])

importance = np.floor(counts.mean(axis=1)).astype(int)
raw_score = (counts * photometric_loss).sum(axis=1)
pruning = (raw_score - raw_score.min()) / (raw_score.max() - raw_score.min())

for name, imp, raw, prune in zip("ABC", importance, raw_score, pruning):
    print(f"G_{name}: importance={imp}  raw={raw:.2f}  pruning={prune:.3f}")

print()
print("densify candidates, importance > 5:",
      [f"G_{name}" for name, imp in zip("ABC", importance) if imp > 5])
print("final prune candidates, pruning > 0.9:",
      [f"G_{name}" for name, prune in zip("ABC", pruning) if prune > 0.9])


**Đọc kết quả:**

- $G_A$: Importance $=\lfloor7.0\rfloor=7 >5$ → ứng viên densify. Pruning $=1.0$.
- $G_B$: Importance $=\lfloor4.0\rfloor=4$ → **không** densify: sai nhiều nhưng **chỉ ở 1 góc nhìn** (tránh nhồi Gaussian cho artefact cục bộ). Pruning $\approx 0.913$.
- $G_C$: Importance $=0$, Pruning $=0.0$ → giữ nguyên.

> **Nghịch lý biểu kiến:** $G_A$ vừa là ứng viên **densify** vừa là ứng viên **prune**. Không mâu thuẫn: hai điểm số dùng ở **hai giai đoạn khác nhau** — densify chạy ở vòng < 15k để *thử thêm chi tiết*, final-prune chạy ở vòng > 15k để *dọn những gì không giúp được*.

## Phần 3: Densification có điều kiện kép

Hàm `densify_and_prune_fastgs` (`scene/gaussian_model.py:468`). Một Gaussian chỉ được nhân bản khi **thoả đồng thời hai điều kiện độc lập**.

### 3.1 — Điều kiện gradient (chọn *ở đâu* cần thêm chi tiết)

$$\text{clone}_i:\ \lVert \bar{g}_i\rVert \ge \tau_{\text{grad}}\ \ \wedge\ \ \max(\text{scale}_i)\le \delta\cdot\text{extent}$$

$$\text{split}_i:\ \lVert \bar{g}^{\text{abs}}_i\rVert \ge \tau_{\text{grad}}^{\text{abs}}\ \ \wedge\ \ \max(\text{scale}_i) > \delta\cdot\text{extent}$$

| Ký hiệu | Ý nghĩa | Tham số CLI (mặc định) |
|---|---|---|
| $\bar{g}_i$ | Gradient vị trí 2D tích luỹ / số lần quan sát (`xyz_gradient_accum / denom`) | `--grad_thresh` (0.0002) |
| $\bar{g}^{\text{abs}}_i$ | Gradient **trị tuyệt đối** tích luỹ (kiểu Abs-GS) — bắt vùng gradient dao động đổi dấu mà tổng gần 0 | `--grad_abs_thresh` (0.0012) |
| $\delta\cdot\text{extent}$ | Ngưỡng kích thước: Gaussian nhỏ thì **clone** (thiếu mật độ), to thì **split** (thiếu độ mịn) | `--dense` (0.001) |

### 3.2 — Điều kiện nhất quán đa góc nhìn (lọc *cái nào* thực sự đáng thêm)

$$\text{metric\_mask}_i = \bigl[\ \text{Importance}_i > 5\ \bigr]$$

```python
# scene/gaussian_model.py:494
metric_mask = importance_score > 5
self.densify_and_clone_fastgs(metric_mask, all_clones)   # AND theo tung phan tu
self.densify_and_split_fastgs(metric_mask, all_splits)
```

**Đây là điểm khác biệt cốt lõi với 3DGS gốc.** 3DGS densify mọi Gaussian có gradient lớn. FastGS thêm phép **AND**: gradient lớn *và* nhiều góc nhìn cùng thấy sai. Hệ quả:

- Gaussian ở vùng đã hội tụ (gradient còn dư nhưng render đã đúng) → Importance thấp → **không nhân bản** → số điểm không phình.
- Gaussian ở artefact chỉ thấy từ 1–2 góc (floater phản chiếu) → Importance thấp → **không được củng cố**.

In [ ]:
N = 12
grad = rng.uniform(0, 0.0006, size=N)
grad_abs = rng.uniform(0, 0.0030, size=N)
scale_max = rng.uniform(0, 0.004, size=N)
importance_score = rng.integers(0, 12, size=N)

grad_thresh, grad_abs_thresh = 0.0002, 0.0012
dense, extent = 0.001, 1.0
size_cut = dense * extent

grad_ok = grad >= grad_thresh
grad_abs_ok = grad_abs >= grad_abs_thresh
is_small = scale_max <= size_cut
is_large = scale_max > size_cut

all_clones = grad_ok & is_small
all_splits = grad_abs_ok & is_large
metric_mask = importance_score > 5

final_clone = all_clones & metric_mask
final_split = all_splits & metric_mask

print(" i  grad  abs  small  imp>5 | 3dgs_clone  fastgs_clone  fastgs_split")
for i in range(N):
    print(f"{i:2d}    {int(grad_ok[i])}    {int(grad_abs_ok[i])}     {int(is_small[i])}"
          f"      {int(metric_mask[i])}   |      {int(all_clones[i])}            {int(final_clone[i])}"
          f"             {int(final_split[i])}")

print()
print(f"3dgs would densify {all_clones.sum() + all_splits.sum()} gaussians")
print(f"fastgs densifies   {final_clone.sum() + final_split.sum()} gaussians")


### 3.3 — Ba tầng pruning

| Khi nào | Điều kiện xoá | Vị trí trong code |
|---|---|---|
| Mỗi lần densify | `opacity < 0.005`, hoặc bán kính màn hình > 20 px, hoặc scale > `0.1·extent` | `gaussian_model.py:499–503` |
| Mỗi lần densify (tuỳ chọn) | Lấy mẫu `0.5 × (số điểm opacity thấp)` để xoá, xác suất tỉ lệ $1/(1-\text{Pruning}_i)$ | `gaussian_model.py:505–518` |
| **Mỗi 3.000 vòng, sau vòng 15k** | `opacity < 0.1` **hoặc** `Pruning_i > 0.9` — `final_prune_fastgs` | `train.py:153`, `gaussian_model.py:533` |

Tầng thứ ba là "dọn dẹp mạnh tay" giai đoạn cuối: sau ~15k vòng mô hình đã cơ bản hội tụ, nên có thể cắt aggressively những Gaussian điểm-pruning cao mà **không giảm chất lượng** (xem thí nghiệm 20k vòng trong bản arXiv).

In [ ]:
def final_prune_fastgs(opacity, pruning_score, min_opacity=0.1):
    return (opacity < min_opacity) | (pruning_score > 0.9)


opacity = rng.uniform(0.0, 1.0, size=N)
pruning_score = rng.uniform(0.0, 1.0, size=N)
removed = final_prune_fastgs(opacity, pruning_score)

print("removed at final prune:", np.where(removed)[0].tolist())
print(f"kept {(~removed).sum()} of {N}")


## Phần 4: Compact box — giảm số tile mỗi splat (`--mult`)

3DGS gán mỗi splat 2D cho **mọi tile 16×16 mà bounding box hình chữ nhật của nó chạm tới**. Bounding box tính theo $3\sigma$ của ellipse là **lỏng** — nhiều tile ở góc hộp gần như không nhận đóng góp nào nhưng vẫn phải vào bước sort + blend.

FastGS nhân bán trục của hộp với hệ số `mult`:

$$\text{half-extent}_{x} = \texttt{mult}\cdot 3\sqrt{\Sigma'_{11}},\qquad
\text{half-extent}_{y} = \texttt{mult}\cdot 3\sqrt{\Sigma'_{22}}$$

| `mult` | Hệ quả |
|---|---|
| `0.5` (mặc định) | Hộp co còn nửa → số cặp (tile, Gaussian) giảm mạnh → sort & rasterize nhanh hơn |
| `0.7` (Tanks&Temples, Deep Blending trong `train_big.sh`) | Cân bằng an toàn hơn khi splat lớn / nền phức tạp |
| → 1.0 | Quay về hành vi 3DGS gốc |

Vì đuôi Gaussian ngoài ~$2\sigma$ đóng góp $\alpha$ rất nhỏ (công thức (4) trong tài liệu toán), cắt bớt rìa hộp gần như **không đổi ảnh** nhưng bỏ được nhiều phép tính. `mult` phải truyền **nhất quán cho cả `train.py` và `render.py`** (thấy rõ trong `train_base.sh`).

In [ ]:
TILE = 16
sigma_x, sigma_y = 22.0, 14.0


def tiles_touched(mult):
    half_x, half_y = mult * 3 * sigma_x, mult * 3 * sigma_y
    tiles_x = np.ceil((2 * half_x) / TILE) + 1
    tiles_y = np.ceil((2 * half_y) / TILE) + 1
    return int(tiles_x * tiles_y)


baseline_tiles = tiles_touched(1.0)
for mult in (1.0, 0.7, 0.5):
    count = tiles_touched(mult)
    print(f"mult={mult}: about {count:3d} tiles per splat "
          f"({count / baseline_tiles:.0%} of the 3dgs bounding box)")


## Phần 5: Tách learning rate cho Spherical Harmonics

3DGS dùng một `feature_lr = 0.0025` cho toàn bộ hệ số SH. FastGS chia đôi:

| Tham số | Điều khiển | Mặc định | Vai trò |
|---|---|---|---|
| `--lowfeature_lr` | `features_dc` — SH bậc 0, tức **màu cơ bản** không phụ thuộc góc nhìn | 0.0025 | Giữ nguyên nhịp cũ |
| `--highfeature_lr` | `features_rest` — SH bậc 1–3, tức **phần màu thay đổi theo góc nhìn** (specular, ánh kim) | 0.005 (tới 0.02–0.04 cho cảnh nhiều phản xạ) | Tăng 2–8× để hội tụ nhanh |

Lý do tách: thành phần bậc thấp mang phần lớn năng lượng màu, dễ bất ổn nếu lr cao; thành phần bậc cao nhỏ và cần nhiều bước để "nở" ra. Cho phần bậc cao một lr lớn hơn giúp màu phụ thuộc góc nhìn **đạt được trong ít vòng lặp hơn** — quan trọng khi tổng ngân sách chỉ ~vài nghìn vòng hiệu dụng.

Xem `train_base.sh`: `garden`, `room`, `counter`, `kitchen`, `bonsai` đặt `--highfeature_lr 0.02`; các cảnh Tanks&Temples đặt `0.04`.

In [ ]:
def steps_to_converge(lr, target=1.0, tol=0.05, max_steps=20000):
    weight = 0.0
    for step in range(1, max_steps + 1):
        weight += lr * (target - weight)
        if abs(target - weight) < tol:
            return step
    return max_steps


for lr in (0.0025, 0.005, 0.02, 0.04):
    print(f"highfeature_lr={lr:<7}: about {steps_to_converge(lr):5d} steps "
          f"to reach 95 percent of the target value")


## Phần 6: Vì sao cộng lại thành "100 giây"?

| Cơ chế | Cắt giảm cái gì | Đòn bẩy |
|---|---|---|
| Điều kiện AND khi densify (Phần 3.2) | Số Gaussian sinh ra ở vùng đã tốt / artefact cục bộ | Ít điểm hơn ⇒ forward + backward + optimizer mỗi vòng rẻ hơn |
| Final-prune theo pruning score (Phần 3.3) | Đuôi dài Gaussian vô ích ở nửa sau huấn luyện | Giảm điểm ⇒ giảm VRAM và thời gian mỗi vòng |
| Compact box `--mult` (Phần 4) | Số cặp (tile, Gaussian) phải sort/blend | Kernel rasterization nhanh hơn ở **mọi** vòng, cả train lẫn render |
| Tách lr SH (Phần 5) | Số vòng cần để màu specular hội tụ | Về đích với ít vòng lặp hiệu dụng hơn |
| Lấy mẫu 10 camera cho scoring (Phần 1) | Chi phí của chính bước scoring | Ước lượng đủ tốt mà không render toàn bộ tập train |

**So sánh tổng quan** (số liệu từ README):

| | 3DGS gốc | FastGS |
|---|---|---|
| Thời gian train | 5–30 phút | **~100 giây** |
| Tiêu chí densify | Chỉ gradient | Gradient **AND** nhất quán đa góc nhìn |
| Kiểm soát số Gaussian | Phình tự do | Chặn ở cả hai đầu (sinh có điều kiện + prune theo score) |
| Tăng tốc so với 3DGS | — | 3.32× so với DashGaussian (Mip-NeRF 360); 15.45× so với 3DGS (Deep Blending) |
| Chất lượng render | chuẩn | ngang ngửa SOTA |

In [ ]:
import matplotlib.pyplot as plt

iters = np.arange(0, 30001, 500)


def grow(spawn_rate, prune_rate, final_prune_frac=0.0):
    count, trajectory = 100_000, []
    for iteration in iters:
        if iteration < 15000:
            count = count * (1 + spawn_rate) * (1 - prune_rate)
        if final_prune_frac and iteration >= 15000 and iteration % 3000 == 0:
            count *= (1 - final_prune_frac)
        trajectory.append(count)
    return np.array(trajectory)


n_3dgs = grow(spawn_rate=0.06, prune_rate=0.01)
n_fastgs = grow(spawn_rate=0.02, prune_rate=0.015, final_prune_frac=0.08)

print(f"{'iter':>7} | {'3dgs':>12} | {'fastgs':>12}")
for iteration, a, b in list(zip(iters, n_3dgs, n_fastgs))[::6]:
    print(f"{iteration:7d} | {a:12,.0f} | {b:12,.0f}")
print()
print(f"final: 3dgs {n_3dgs[-1]:,.0f} points, fastgs {n_fastgs[-1]:,.0f} points "
      f"({n_fastgs[-1] / n_3dgs[-1]:.0%})")

plt.figure(figsize=(7, 4))
plt.plot(iters, n_3dgs / 1e6, label="3dgs, gradient only")
plt.plot(iters, n_fastgs / 1e6, label="fastgs, AND condition plus final prune")
plt.xlabel("iteration")
plt.ylabel("gaussians (millions)")
plt.title("Gaussian count trajectory")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Kết luận:** ý tưởng trung tâm của FastGS là thay câu hỏi *"Gaussian này có gradient lớn không?"* bằng *"Nhiều camera có cùng đồng ý rằng chỗ này đang sai không?"*.

Tín hiệu nhất quán đa góc nhìn vừa **rẻ để tính** (10 lần render phụ), vừa **chọn lọc hơn nhiều** — nên FastGS thêm Gaussian đúng chỗ, bỏ Gaussian đúng lúc, và giữ mỗi bước rasterization gọn nhẹ.

Bản văn xuôi đầy đủ của Phần 1–6: [DOCS/fastgs-acceleration-method.md](DOCS/fastgs-acceleration-method.md) — kèm **ba điều mã nguồn nói mà README không nói** (lịch optimizer đóng đinh theo 30k vòng, `--antialiasing` là cờ chết, `densification_interval` là lever thời gian bị bỏ quên). Paper: [arXiv:2511.04283](https://arxiv.org/abs/2511.04283).

Hết phần lý thuyết. **Phần 7 trở đi chạy mô hình thật trên GPU.**

---

# Phần 7: Huấn luyện thật trên Google Colab (T4 free)

Sáu phần trên là **mô phỏng thu nhỏ**. Phần này chạy **mô hình thật** trên GPU **T4 miễn phí** của Colab, đáp ứng bốn yêu cầu:

1. **Viết chi tiết vòng lặp huấn luyện** ngay trong notebook (không gọi `train.py` như hộp đen) để cắm được móc theo dõi.
2. **Hiện tiến trình bằng một điểm số tổng hợp**, đo **mỗi 1000 vòng**:

$$\boxed{\ \mathrm{Score}=0.4\,(1-\mathrm{LPIPS})+0.3\,\mathrm{SSIM}+0.3\,\widehat{\mathrm{PSNR}}\ }
\qquad
\widehat{\mathrm{PSNR}}=\operatorname{clamp}\!\Big(\tfrac{\mathrm{PSNR}}{\mathrm{PSNR}_{\max}},\,0,\,1\Big)$$

   in kèm **mức tăng/giảm mỗi 1000 vòng** ($\Delta_{1k}$) cho từng thành phần.
3. **Sau huấn luyện**: render ảnh, in thông số, vẽ sơ đồ bằng matplotlib (đường cong, phương trình 2D/3D, ảnh ví dụ GT / render / bản đồ lỗi).
4. **Tối ưu RAM cho Colab free** để train xong không bị RAM đầy → vẫn tải được mô hình về; kèm **thống kê kiểu data engineer**.

| Tài nguyên Colab free | Hạn mức thực tế | Rủi ro chính |
|---|---|---|
| GPU T4 | ~15 GB VRAM | OOM nếu số Gaussian phình / ảnh độ phân giải cao |
| RAM hệ thống | ~12.7 GB | **train xong RAM còn kẹt → `zip` + `files.download` OOM → mất mô hình** |
| Đĩa | ~78–110 GB | thường không phải nút thắt |

**Cách dùng:** đặt runtime là **T4 GPU** rồi **Runtime → Run all**. Toàn bộ tham số nằm ở ô cấu hình 7.0; mọi ô nặng đều tự bỏ qua khi thiếu điều kiện, nên Run all không bị đứt giữa chừng.

**Thứ tự:** 7.0 cấu hình → 7.1 cài đặt → 7.2 dữ liệu + hồ sơ → 7.3 định nghĩa Score → 7.4 huấn luyện → 7.5 dọn RAM → 7.6 render + metrics (tiến trình con) → 7.7 biểu đồ → 7.8 thống kê → 7.9 tải mô hình.

> Toàn bộ chống-OOM RAM nằm ở **7.4 – 7.5 – 7.6 – 7.9**: lưu `.ply` ra đĩa sớm, dọn RAM ngay sau train, đẩy render/metrics sang **tiến trình con**, và `zip` **ra đĩa** thay vì vào RAM.


## 7.0 — Cấu hình: sửa ở đây, một chỗ duy nhất

Notebook được thiết kế để **Runtime → Run all** chạy hết từ đầu tới cuối trên T4 free mà không phải sửa gì. Mọi tham số nằm gọn trong ô dưới.

| Công tắc | Mặc định | Ý nghĩa |
|---|---|---|
| `USE_DEMO_DATASET` | `True` | Tự tải scene `tandt/truck` (~650 MB) nếu `SOURCE_PATH` chưa có. Đặt `False` khi bạn tự trỏ dữ liệu |
| `ITERATIONS` | `7000` | Đủ để Run all kết thúc trong khoảng 15–25 phút. **Đặt `30000` cho kết quả đầy đủ** — xem mục 2.1 của [DOCS/colab-t4-guide.md](DOCS/colab-t4-guide.md) về lý do không nên để dưới 20000 |
| `RUN_TRAINING` | `True` | Tắt để chỉ đọc lý thuyết và chạy các ô mô phỏng |
| `RUN_ABLATION` | `False` | Phần 8 chạy **thêm 3 lượt train**. Mặc định tắt để Run all không kéo dài hàng giờ |
| `DOWNLOAD_MODEL_TO_BROWSER` | `True` | Sau khi train xong, `files.download` đẩy file `.zip` mô hình về máy bạn |
| `SAVE_MODEL_TO_DRIVE` | `False` | Chép `.zip` vào Google Drive. Tự bật khi file ≥ 200 MB |

Mọi ô nặng đều có rào chắn: thiếu dữ liệu hoặc thiếu kết quả của bước trước thì ô **in thông báo rồi bỏ qua**, không ném lỗi làm đứt Run all.


In [ ]:
REPO_URL = "https://github.com/KietAnhCS/fastgs-lite.git"
REPO_DIR = "/content/fastgs-lite"

USE_DEMO_DATASET = True
DEMO_URL = "https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip"
DEMO_ROOT = "/content/data"

SOURCE_PATH = "/content/data/tandt/truck"
IMAGES_DIR = "images"
RESOLUTION = 2
LLFFHOLD = 8

MODEL_PATH = "/content/output/truck"
ITERATIONS = 7000

RUN_TRAINING = True
RUN_ABLATION = False
DOWNLOAD_MODEL_TO_BROWSER = True
SAVE_MODEL_TO_DRIVE = False

PSNR_MAX = 30.0
SCORE_EVERY = 1000
EVAL_VIEWS = 6
RAM_SOFT_LIMIT_GB = 10.5
MULT = 0.5

for key in ("REPO_DIR", "SOURCE_PATH", "MODEL_PATH", "RESOLUTION", "ITERATIONS",
            "RUN_TRAINING", "RUN_ABLATION", "DOWNLOAD_MODEL_TO_BROWSER",
            "SAVE_MODEL_TO_DRIVE"):
    print(f"{key:26s} = {globals()[key]}")


In [ ]:
import os, gc, time, json
import psutil, torch

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


def _gb(x):
    return x / (1024 ** 3)


def mem():
    vm = psutil.virtual_memory()
    d = dict(ram_used_gb=_gb(vm.used), ram_total_gb=_gb(vm.total), ram_pct=vm.percent)
    if torch.cuda.is_available():
        d["vram_alloc_gb"] = _gb(torch.cuda.memory_allocated())
        d["vram_reserved_gb"] = _gb(torch.cuda.memory_reserved())
        d["vram_total_gb"] = _gb(torch.cuda.get_device_properties(0).total_memory)
    return d


def show_mem(tag=""):
    m = mem()
    line = f"[MEM {tag:<22}] RAM {m['ram_used_gb']:5.2f}/{m['ram_total_gb']:4.1f} GB ({m['ram_pct']:4.1f}%)"
    if "vram_alloc_gb" in m:
        line += (f" | VRAM {m['vram_alloc_gb']:5.2f} alloc / {m['vram_reserved_gb']:5.2f} reserved"
                 f" / {m['vram_total_gb']:4.1f} GB")
    print(line)
    return m


print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "no GPU detected -> Runtime > Change runtime type > T4 GPU")
show_mem("startup")


## 7.1 — Cài đặt (một lần mỗi phiên Colab)

Sửa `REPO_URL` thành fork của bạn. Các submodule CUDA (`diff-gaussian-rasterization_fastgs`, `fused-ssim`) biên dịch ~3–5 phút cho lần đầu; các lần sau bỏ qua nhờ cờ `/content/.deps_ok`.

> Runtime → Change runtime type → **T4 GPU** trước khi chạy.


In [ ]:
import os

if not os.path.isdir(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

DEPS_FLAG = "/content/.deps_ok"
if not os.path.exists(DEPS_FLAG):
    !pip -q install plyfile psutil pandas matplotlib tqdm
    !pip -q install ./submodules/diff-gaussian-rasterization_fastgs
    !pip -q install ./submodules/fused-ssim
    !pip -q install ./submodules/simple-knn
    open(DEPS_FLAG, "w").close()

import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")


## 7.2 — Nạp dữ liệu & hồ sơ dữ liệu (data engineer #1)

Ô đầu lo phần dữ liệu: nếu `SOURCE_PATH` chưa tồn tại và `USE_DEMO_DATASET = True` thì tự tải `tandt_db.zip` (~650 MB, link chính thức của INRIA) và giải nén, lấy scene `tandt/truck`. Tải hỏng thì nó **in hướng dẫn rồi đặt `DATA_READY = False`**, các ô sau tự bỏ qua chứ không ném lỗi.

Muốn dùng dữ liệu của bạn: đặt `USE_DEMO_DATASET = False` và trỏ `SOURCE_PATH` tới thư mục COLMAP có `images/` + `sparse/0/`.

Ô thứ hai lập **hồ sơ dữ liệu trước khi train**: số ảnh, phân giải, dung lượng, tách train/test (`llffhold = 8` — cứ 8 ảnh lấy 1 ảnh test, giống `readColmapSceneInfo`), số điểm COLMAP, và phân giải hiệu dụng sau `-r`. Đây là bước "biết mình đang train trên cái gì" và là đầu vào cho báo cáo thống kê ở 7.8.


In [ ]:
import os, subprocess

DATA_READY = os.path.isdir(os.path.join(SOURCE_PATH, "sparse"))

if DATA_READY:
    print("dataset already present:", SOURCE_PATH)
elif USE_DEMO_DATASET:
    os.makedirs(DEMO_ROOT, exist_ok=True)
    archive = os.path.join(DEMO_ROOT, "tandt_db.zip")
    if not os.path.exists(archive) or os.path.getsize(archive) < 10_000_000:
        print("downloading demo dataset, about 650 MB")
        subprocess.run(["wget", "-q", "--show-progress", "-O", archive, DEMO_URL], check=False)
    if os.path.exists(archive) and os.path.getsize(archive) > 10_000_000:
        subprocess.run(["unzip", "-q", "-o", archive, "-d", DEMO_ROOT], check=False)
        DATA_READY = os.path.isdir(os.path.join(SOURCE_PATH, "sparse"))
    if not DATA_READY:
        print("demo download failed")
        print("set SOURCE_PATH in the config cell to your own COLMAP folder")
else:
    print("SOURCE_PATH not found:", SOURCE_PATH)
    print("set USE_DEMO_DATASET = True, or point SOURCE_PATH at a COLMAP dataset")

print("DATA_READY =", DATA_READY)


In [ ]:
import os, json, numpy as np, pandas as pd
from PIL import Image


def profile_dataset(source_path, images_dir=IMAGES_DIR, llffhold=LLFFHOLD, downscale=RESOLUTION):
    image_root = os.path.join(source_path, images_dir)
    files = sorted(f for f in os.listdir(image_root) if f.lower().endswith((".jpg", ".jpeg", ".png")))
    rows = []
    for i, name in enumerate(files):
        path = os.path.join(image_root, name)
        with Image.open(path) as im:
            width, height = im.size
        rows.append(dict(idx=i, name=name, width=width, height=height,
                         megapixel=round(width * height / 1e6, 2),
                         file_MB=round(os.path.getsize(path) / 1e6, 3),
                         split="test" if i % llffhold == 0 else "train"))
    df = pd.DataFrame(rows)

    colmap_points = None
    for candidate in ("sparse/0/points3D.bin", "sparse/0/points3D.txt", "sparse/points3D.bin"):
        path = os.path.join(source_path, candidate)
        if not os.path.exists(path):
            continue
        try:
            if path.endswith(".txt"):
                colmap_points = sum(1 for line in open(path) if line.strip() and not line.startswith("#"))
            else:
                from scene.colmap_loader import read_points3D_binary
                colmap_points = int(read_points3D_binary(path)[0].shape[0])
        except Exception as exc:
            print("could not read points3D:", exc)
        break

    summary = dict(
        source=source_path,
        n_images=len(df),
        n_train=int((df.split == "train").sum()),
        n_test=int((df.split == "test").sum()),
        res_min=f"{int(df.width.min())}x{int(df.height.min())}",
        res_max=f"{int(df.width.max())}x{int(df.height.max())}",
        megapixel_mean=round(float(df.megapixel.mean()), 2),
        images_disk_MB=round(float(df.file_MB.sum()), 1),
        colmap_points=colmap_points,
        effective_train_res=f"{int(df.width.median() // downscale)}x{int(df.height.median() // downscale)}",
    )
    return df, summary


ds_df, ds_summary = None, None
if DATA_READY:
    ds_df, ds_summary = profile_dataset(SOURCE_PATH)
    print(json.dumps(ds_summary, indent=2, default=str))
    display(ds_df.head(10))
else:
    print("skipped: DATA_READY is False")


## 7.3 — Điểm số tiến trình: một con số cho "đang tốt lên hay xấu đi"

Loss huấn luyện (L1 + DSSIM) tụt rất nhanh rồi phẳng — khó nhìn tiến bộ ở nửa sau. Ta gộp ba thước đo chuẩn của novel-view synthesis thành **một** số trong $[0,1]$, càng cao càng tốt:

$$\mathrm{Score}=0.4\,(1-\mathrm{LPIPS})+0.3\,\mathrm{SSIM}+0.3\,\widehat{\mathrm{PSNR}},
\qquad
\widehat{\mathrm{PSNR}}=\operatorname{clamp}\!\Big(\tfrac{\mathrm{PSNR}}{\mathrm{PSNR}_{\max}},\,0,\,1\Big)$$

| Ký hiệu | Ý nghĩa | Miền | Đóng góp vào Score |
|---|---|---|---|
| $\mathrm{LPIPS}$ | khoảng cách tri giác (mạng VGG); **thấp** = giống | $[0,1]$ | $0.4\,(1-\mathrm{LPIPS})$ |
| $\mathrm{SSIM}$ | tương đồng cấu trúc; **cao** = giống | $[0,1]$ | $0.3\,\mathrm{SSIM}$ |
| $\mathrm{PSNR}$ | tỉ số tín hiệu / nhiễu (dB), không có trần | $[0,\infty)$ | — |
| $\widehat{\mathrm{PSNR}}$ | `torch.clamp(psnr_val / psnr_max, 0.0, 1.0)` | $[0,1]$ | $0.3\,\widehat{\mathrm{PSNR}}$ |
| $\mathrm{PSNR}_{\max}$ | mốc bão hoà (mặc định **30 dB**) | hằng | — |

**Vì sao chuẩn hoá PSNR.** LPIPS và SSIM đã nằm sẵn trong $[0,1]$; PSNR thì không. Để nguyên, một cảnh dễ (PSNR 35) sẽ áp đảo tổng và Score mất khả năng so sánh. `clamp(psnr/psnr_max, 0, 1)` kéo nó về cùng thang **và triệt tiêu phần thưởng cho việc vượt xa ngưỡng "đủ tốt"** — đúng tinh thần FastGS: về đích nhanh, không nhồi thêm.

**Đọc $\Delta_{1k}$.** Sau mỗi 1000 vòng, in $\Delta_{1k}\mathrm{Score}=\mathrm{Score}_t-\mathrm{Score}_{t-1000}$ (và $\Delta_{1k}$ từng thành phần). Chuỗi $\Delta$ dương, co dần về 0 = hội tụ lành mạnh; $\Delta$ âm kéo dài = bắt đầu overfit hoặc prune quá tay.


In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt


def composite_score(psnr_val, ssim_val, lpips_val, psnr_max=None):
    psnr_max = PSNR_MAX if psnr_max is None else psnr_max
    psnr_norm = torch.clamp(torch.as_tensor(float(psnr_val)) / psnr_max, 0.0, 1.0)
    ssim_t = torch.as_tensor(float(ssim_val))
    lpips_t = torch.as_tensor(float(lpips_val))
    score = 0.4 * (1.0 - lpips_t) + 0.3 * ssim_t + 0.3 * psnr_norm
    return float(score), float(psnr_norm)


LPIPS_FIXED = 0.20
fig = plt.figure(figsize=(14, 4.4))

ax0 = fig.add_subplot(1, 3, 1)
ax0.axis("off")
ax0.text(0.5, 0.74, r"$\mathrm{Score}=0.4\,(1-\mathrm{LPIPS})+0.3\,\mathrm{SSIM}+0.3\,\widehat{\mathrm{PSNR}}$",
         ha="center", va="center", fontsize=12.5)
ax0.text(0.5, 0.46, r"$\widehat{\mathrm{PSNR}}=\mathrm{clamp}\!\left(\dfrac{\mathrm{PSNR}}{\mathrm{PSNR}_{\max}},\ 0,\ 1\right)$",
         ha="center", va="center", fontsize=12)
ax0.text(0.5, 0.16, f"$\\mathrm{{PSNR}}_{{\\max}} = {PSNR_MAX:.0f}$ dB", ha="center", va="center", fontsize=11)
ax0.set_title("Definition")

ax1 = fig.add_subplot(1, 3, 2)
psnr_axis = np.linspace(10, 35, 240)
for ssim_level in (0.50, 0.70, 0.85, 0.95):
    curve = 0.4 * (1 - LPIPS_FIXED) + 0.3 * ssim_level + 0.3 * np.clip(psnr_axis / PSNR_MAX, 0, 1)
    ax1.plot(psnr_axis, curve, label=f"SSIM = {ssim_level:.2f}")
ax1.axvline(PSNR_MAX, ls="--", c="gray", lw=1)
ax1.set_xlabel("PSNR (dB)")
ax1.set_ylabel("Score")
ax1.set_title(f"LPIPS fixed at {LPIPS_FIXED}")
ax1.legend(fontsize=8)
ax1.grid(alpha=.3)

ax2 = fig.add_subplot(1, 3, 3, projection="3d")
ssim_grid, psnr_grid = np.meshgrid(np.linspace(0, 1, 40), np.linspace(0, 1, 40))
score_grid = 0.4 * (1 - LPIPS_FIXED) + 0.3 * ssim_grid + 0.3 * psnr_grid
surface = ax2.plot_surface(ssim_grid, psnr_grid, score_grid, cmap="viridis", alpha=.92)
ax2.set_xlabel("SSIM")
ax2.set_ylabel(r"$\widehat{PSNR}$")
ax2.set_zlabel("Score")
ax2.set_title(f"Score surface, LPIPS = {LPIPS_FIXED}")
fig.colorbar(surface, ax=ax2, shrink=.55, pad=.1)

plt.tight_layout()
plt.show()

print("PSNR=27, SSIM=0.90, LPIPS=0.15 ->", composite_score(27, 0.90, 0.15))
print("PSNR=33, SSIM=0.95, LPIPS=0.08 ->", composite_score(33, 0.95, 0.08))


## 7.4 — Vòng lặp huấn luyện `train_fastgs` + móc theo dõi Score

`train_fastgs` là bản viết lại của `training()` trong `train.py`, **giữ nguyên** ba cơ chế FastGS đã mô phỏng ở Phần 1–5:

- densify với điều kiện **AND** đa góc nhìn khi vòng `< densify_until_iter` — `compute_gaussian_score_fastgs(..., DENSIFY=True)` → `densify_and_prune_fastgs`;
- `final_prune_fastgs` mỗi 3000 vòng trong khoảng (15k, 30k);
- compact box `--mult` và tách lr SH đi qua `opt`.

**Thêm vào để hiện tiến trình:**

| Thành phần | Chi tiết |
|---|---|
| `evaluate_on_holdout` mỗi `score_every=1000` vòng | render `eval_views` camera hold-out ở **scale 1.0** → PSNR/SSIM/LPIPS → `Score` + `psnr_norm` |
| In `dScore/1k` | $\Delta_{1k}$ Score so với mốc trước, kèm PSNR/SSIM/LPIPS, số Gaussian, loss, RAM, VRAM |
| LPIPS live dùng `alex` | nhanh ~3× so với `vgg`; **con số báo cáo** vẫn lấy từ `metrics.py` (vgg) ở 7.6/7.8 |
| Chống RAM | `empty_cache()` mỗi mốc; `gc.collect()` khi vượt `RAM_SOFT_LIMIT_GB`; `del` tensor mỗi vòng |
| Lưu sớm | `scene.save(iteration)` ghi `point_cloud.ply` ra đĩa ngay tại `save_iterations` |

**Các cờ nâng cấp (mặc định TẮT — Phần 7 là baseline preset A, Phần 8 mới bật):**

| Cờ | Ý tưởng gốc | Mặc định |
|---|---|---|
| `resolution_scales`, `resolution_switch_fracs` | coarse-to-fine (DashGaussian) | `(1.0,)`, `()` |
| `gaussian_budget`, `budget_power`, `budget_until_frac` | ngân sách primitive (Taming-3DGS / 3DGS-MCMC) | `None` |
| `mcmc_noise`, `mcmc_gate_k`, `mcmc_gate_tau` | nhiễu SGLD có cổng theo opacity (3DGS-MCMC) | `0.0` |
| `lambda_opacity`, `lambda_scale` | phạt L1 opacity & scale (3DGS-MCMC) | `0.0` |

Hold-out = tập test (khi `--eval`); nếu không có thì lấy `train[::8]`, nhãn `train_subset` và điểm sẽ lạc quan hơn thực tế.


In [ ]:
import torch
from gaussian_renderer import render_fastgs
from utils.image_utils import psnr as psnr_fn
from fused_ssim import fused_ssim as fast_ssim
from lpipsPyTorch import lpips as lpips_fn


@torch.no_grad()
def evaluate_on_holdout(gaussians, cams, pipe, background, mult,
                        max_views=6, psnr_max=PSNR_MAX, lpips_net="alex"):
    if len(cams) == 0:
        return None
    cams = list(cams)[:max_views]
    psnr_sum = ssim_sum = lpips_sum = 0.0
    for cam in cams:
        rendered = torch.clamp(render_fastgs(cam, gaussians, pipe, background, mult)["render"], 0.0, 1.0)
        gt = torch.clamp(cam.original_image.to("cuda"), 0.0, 1.0)
        psnr_sum += psnr_fn(rendered, gt).mean().item()
        ssim_sum += fast_ssim(rendered.unsqueeze(0), gt.unsqueeze(0)).item()
        lpips_sum += lpips_fn(rendered, gt, net_type=lpips_net).mean().item()
        del rendered, gt
    n = len(cams)
    psnr_val, ssim_val, lpips_val = psnr_sum / n, ssim_sum / n, lpips_sum / n
    score, psnr_norm = composite_score(psnr_val, ssim_val, lpips_val, psnr_max)
    torch.cuda.empty_cache()
    return dict(psnr=psnr_val, ssim=ssim_val, lpips=lpips_val,
                psnr_norm=psnr_norm, score=score, n_views=n)


In [ ]:
import time, gc, torch
from random import randint
from tqdm.auto import tqdm

from scene import Scene, GaussianModel
from utils.loss_utils import l1_loss
from fused_ssim import fused_ssim as fast_ssim
from gaussian_renderer import render_fastgs
from utils.fast_utils import compute_gaussian_score_fastgs, sampling_cameras


def make_resolution_schedule(iterations, scales=(1.0,), switch_fracs=()):
    bounds = [int(f * iterations) for f in switch_fracs]

    def schedule(iteration):
        for bound, scale in zip(bounds, scales[:-1]):
            if iteration < bound:
                return scale
        return scales[-1]

    return schedule


def make_budget_schedule(n_start, n_max, iterations, until_frac=0.5, power=1.0):
    end = max(1, int(until_frac * iterations))

    def schedule(iteration):
        t = min(1.0, iteration / end)
        return int(n_start + (n_max - n_start) * (t ** power))

    return schedule


def enforce_budget(gaussians, budget):
    n = gaussians._xyz.shape[0]
    if budget is None or n <= budget:
        return 0
    opacity = gaussians.get_opacity.squeeze(-1)
    drop = torch.zeros(n, dtype=torch.bool, device=opacity.device)
    drop[torch.topk(opacity, n - budget, largest=False).indices] = True
    gaussians.prune_points(drop)
    return n - budget


def train_fastgs(dataset, opt, pipe,
                 score_every=1000, eval_views=6, psnr_max=PSNR_MAX, lpips_net="alex",
                 save_iterations=(30000,), ram_soft_limit_gb=RAM_SOFT_LIMIT_GB,
                 resolution_scales=(1.0,), resolution_switch_fracs=(),
                 gaussian_budget=None, budget_power=1.0, budget_until_frac=0.5,
                 mcmc_noise=0.0, mcmc_gate_k=100.0, mcmc_gate_tau=0.005,
                 lambda_opacity=0.0, lambda_scale=0.0,
                 tag="run"):
    scales = tuple(dict.fromkeys(list(resolution_scales) + [1.0]))
    gaussians = GaussianModel(dataset.sh_degree, opt.optimizer_type)
    scene = Scene(dataset, gaussians, resolution_scales=list(scales))
    gaussians.training_setup(opt)

    bg_color = [1, 1, 1] if dataset.white_background else [0, 0, 0]
    background = torch.tensor(bg_color, dtype=torch.float32, device="cuda")

    res_schedule = make_resolution_schedule(opt.iterations, scales, resolution_switch_fracs)
    budget_schedule = None
    if gaussian_budget is not None:
        budget_schedule = make_budget_schedule(gaussians._xyz.shape[0], gaussian_budget,
                                               opt.iterations, budget_until_frac, budget_power)

    test_cams = scene.getTestCameras(1.0)
    holdout = list(test_cams) if len(test_cams) else scene.getTrainCameras(1.0)[::8]
    holdout_kind = "test" if len(test_cams) else "train_subset"
    print(f"[{tag}] holdout={len(holdout)} ({holdout_kind}) | scales={scales} | budget={gaussian_budget}")

    history, prev = [], None
    viewpoint_stack, viewpoint_indices = [], []
    current_scale = None
    ema_loss = 0.0
    start_time = time.time()
    torch.cuda.reset_peak_memory_stats()

    pbar = tqdm(range(1, opt.iterations + 1), desc=f"train[{tag}]")
    for iteration in pbar:
        position_lr = gaussians.update_learning_rate(iteration)
        if iteration % 1000 == 0:
            gaussians.oneupSHdegree()

        scale = res_schedule(iteration)
        if scale != current_scale:
            current_scale = scale
            viewpoint_stack, viewpoint_indices = [], []
            gaussians.max_radii2D.zero_()
            tqdm.write(f"[{tag}] iteration {iteration}: resolution_scale -> {scale}")
        train_cams = scene.getTrainCameras(current_scale)

        if not viewpoint_stack:
            viewpoint_stack = train_cams.copy()
            viewpoint_indices = list(range(len(viewpoint_stack)))
        pick = randint(0, len(viewpoint_indices) - 1)
        cam = viewpoint_stack.pop(pick)
        viewpoint_indices.pop(pick)

        bg = torch.rand(3, device="cuda") if opt.random_background else background
        render_pkg = render_fastgs(cam, gaussians, pipe, bg, opt.mult)
        image = render_pkg["render"]
        viewspace_points = render_pkg["viewspace_points"]
        visibility = render_pkg["visibility_filter"]
        radii = render_pkg["radii"]

        gt = cam.original_image.cuda()
        ll1 = l1_loss(image, gt)
        ssim_value = fast_ssim(image.unsqueeze(0), gt.unsqueeze(0))
        loss = (1.0 - opt.lambda_dssim) * ll1 + opt.lambda_dssim * (1.0 - ssim_value)
        if lambda_opacity > 0.0:
            loss = loss + lambda_opacity * gaussians.get_opacity.mean()
        if lambda_scale > 0.0:
            loss = loss + lambda_scale * gaussians.get_scaling.mean()
        loss.backward()

        with torch.no_grad():
            ema_loss = 0.4 * loss.item() + 0.6 * ema_loss

            if iteration < opt.densify_until_iter:
                gaussians.max_radii2D[visibility] = torch.max(gaussians.max_radii2D[visibility], radii[visibility])
                gaussians.add_densification_stats(viewspace_points, visibility)
                if iteration > opt.densify_from_iter and iteration % opt.densification_interval == 0:
                    size_threshold = 20 if (iteration > opt.opacity_reset_interval and current_scale == 1.0) else None
                    camlist = sampling_cameras(train_cams.copy())
                    importance, pruning = compute_gaussian_score_fastgs(
                        camlist, gaussians, pipe, bg, opt, DENSIFY=True)
                    gaussians.densify_and_prune_fastgs(
                        max_screen_size=size_threshold, min_opacity=0.005,
                        extent=scene.cameras_extent, radii=radii, args=opt,
                        importance_score=importance, pruning_score=pruning)
                    if budget_schedule is not None:
                        removed = enforce_budget(gaussians, budget_schedule(iteration))
                        if removed:
                            tqdm.write(f"[{tag}] iteration {iteration}: budget pruned {removed:,}")
                if iteration % opt.opacity_reset_interval == 0 or (
                        dataset.white_background and iteration == opt.densify_from_iter):
                    gaussians.reset_opacity()

            if iteration % 3000 == 0 and 15_000 < iteration < 30_000:
                camlist = sampling_cameras(scene.getTrainCameras(1.0).copy())
                _, pruning = compute_gaussian_score_fastgs(camlist, gaussians, pipe, bg, opt)
                gaussians.final_prune_fastgs(min_opacity=0.1, pruning_score=pruning)

            if iteration < opt.iterations:
                if opt.optimizer_type == "default":
                    gaussians.optimizer_step(iteration)
                else:
                    gaussians.optimizer.step(radii > 0, radii.shape[0])
                    gaussians.optimizer.zero_grad(set_to_none=True)

            if mcmc_noise > 0.0 and iteration < opt.densify_until_iter:
                opacity = gaussians.get_opacity.squeeze(-1)
                gate = torch.sigmoid(-mcmc_gate_k * (opacity - mcmc_gate_tau)).unsqueeze(-1)
                step = mcmc_noise * (position_lr if position_lr else opt.position_lr_init)
                gaussians._xyz.add_(torch.randn_like(gaussians._xyz) * gaussians.get_scaling * gate * step)

            if iteration % score_every == 0 or iteration == opt.iterations:
                torch.cuda.empty_cache()
                evaluation = evaluate_on_holdout(gaussians, holdout, pipe, background,
                                                 opt.mult, eval_views, psnr_max, lpips_net)
                usage = mem()
                row = dict(iter=iteration, n_gauss=int(gaussians._xyz.shape[0]),
                           scale=current_scale, ema_loss=ema_loss,
                           elapsed_s=time.time() - start_time,
                           ram_gb=usage["ram_used_gb"], vram_gb=usage.get("vram_alloc_gb", 0.0),
                           **(evaluation or {}))
                if prev is not None and evaluation is not None:
                    for key in ("score", "psnr", "ssim", "lpips"):
                        row[f"d_{key}"] = row[key] - prev[key]
                history.append(row)
                if evaluation is not None:
                    prev = row
                    delta = row.get("d_score")
                    tqdm.write(
                        f"iter {iteration:6d} | dScore/1k {('%+.4f' % delta) if delta is not None else '   n/a '}"
                        f" | Score {row['score']:.4f}"
                        f" | PSNR {row['psnr']:5.2f} ({row['psnr_norm']:.3f})"
                        f" SSIM {row['ssim']:.4f} LPIPS {row['lpips']:.4f}"
                        f" | G {row['n_gauss']:>9,} | loss {ema_loss:.4f}"
                        f" | RAM {row['ram_gb']:.1f} VRAM {row['vram_gb']:.1f} GB")
                if usage["ram_used_gb"] > ram_soft_limit_gb:
                    gc.collect()
                    torch.cuda.empty_cache()
                    tqdm.write(f"[{tag}] RAM {usage['ram_used_gb']:.2f} GB above limit, collected")

            if iteration in save_iterations:
                scene.save(iteration)
                tqdm.write(f"[{tag}] saved point cloud at iteration {iteration}")

            del render_pkg, image, gt, viewspace_points, visibility, radii, loss, ll1, ssim_value

    pbar.close()
    total_time = time.time() - start_time
    peak_vram = _gb(torch.cuda.max_memory_allocated())
    n_gauss = int(gaussians._xyz.shape[0])
    print(f"[{tag}] done: {n_gauss:,} gaussians | {total_time:.1f}s"
          f" ({total_time / opt.iterations * 1000:.1f}s per 1k iterations) | peak VRAM {peak_vram:.2f} GB")

    return (dict(history=history, model_path=scene.model_path, holdout_kind=holdout_kind,
                 total_time=total_time, peak_vram_gb=peak_vram, n_gauss=n_gauss, tag=tag),
            gaussians, scene)


In [ ]:
import os
from argparse import ArgumentParser, Namespace
from arguments import ModelParams, PipelineParams, OptimizationParams


def preset_a_args(iterations):
    return [
        "--iterations", str(iterations),
        "--densification_interval", "500",
        "--lambda_dssim", "0.25",
        "--highfeature_lr", "0.02",
        "--loss_thresh", "0.07",
        "--grad_abs_thresh", "0.0012",
    ]


def build_args(model_path, extra):
    parser = ArgumentParser()
    lp = ModelParams(parser)
    op = OptimizationParams(parser)
    pp = PipelineParams(parser)
    argv = ["-s", SOURCE_PATH, "-m", model_path, "-i", IMAGES_DIR,
            "-r", str(RESOLUTION), "--eval"] + list(extra)
    args = parser.parse_args(argv)
    os.makedirs(model_path, exist_ok=True)
    with open(os.path.join(model_path, "cfg_args"), "w") as handle:
        handle.write(str(Namespace(**vars(args))))
    return args, lp, op, pp


print("preset A:", " ".join(preset_a_args(30000)))


In [ ]:
import os, pandas as pd
from utils.general_utils import safe_state

result = None
if RUN_TRAINING and DATA_READY:
    args, lp, op, pp = build_args(MODEL_PATH, preset_a_args(ITERATIONS))
    safe_state(False)
    show_mem("before training")

    result, gaussians, scene = train_fastgs(
        lp.extract(args), op.extract(args), pp.extract(args),
        score_every=SCORE_EVERY,
        eval_views=EVAL_VIEWS,
        save_iterations=(ITERATIONS,),
        psnr_max=PSNR_MAX,
        resolution_scales=(1.0,),
        tag="presetA",
    )
    history_frame = pd.DataFrame(result["history"])
    history_frame.to_csv(os.path.join(MODEL_PATH, "score_history.csv"), index=False)
    display(history_frame[["iter", "score", "psnr", "ssim", "lpips", "n_gauss", "ram_gb", "vram_gb"]].round(4))
else:
    print(f"skipped: RUN_TRAINING={RUN_TRAINING}, DATA_READY={DATA_READY}")


## 7.5 — Giải phóng RAM ngay sau huấn luyện (điểm mấu chốt)

**Bẫy kinh điển trên Colab free:** train xong, kernel vẫn giữ `gaussians` (mọi tensor tham số + trạng thái Adam), `scene` (toàn bộ ảnh GT đã nạp vào bộ nhớ), đồ thị autograd, cache CUDA — RAM ~10–12 GB. Đúng lúc đó bạn gọi `shutil.make_archive` / `files.download` → Python giữ thêm một bản copy trong RAM → **OOM, kernel chết, mất mô hình dù `.ply` đã nằm trên đĩa**.

Cách chặn: `.ply` đã được `scene.save()` ghi ra `MODEL_PATH` **trong lúc train** rồi. Giờ chỉ việc **xoá sạch vật thể lớn khỏi RAM**, giữ lại đúng `MODEL_PATH` (một chuỗi) và `result["history"]` (một list nhỏ).


In [ ]:
import gc, os, torch

show_mem("before cleanup")

for name in ("gaussians", "scene"):
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
show_mem("after cleanup")

PLY_PATH = os.path.join(MODEL_PATH, f"point_cloud/iteration_{ITERATIONS}/point_cloud.ply")
HAS_MODEL = os.path.exists(PLY_PATH)

if HAS_MODEL:
    print(f"point cloud: {PLY_PATH} ({os.path.getsize(PLY_PATH) / 1e6:.1f} MB)")
    if result:
        print(f"history points: {len(result['history'])} | training time: {result['total_time']:.1f}s")
else:
    print("no trained model on disk, the remaining cells will be skipped")


## 7.6 — Render ảnh & tính thông số trong **tiến trình con**

`render.py` và `metrics.py` (metrics nạp mạng VGG cho LPIPS, ~500 MB) chạy bằng `!python …`. Chạy ở **tiến trình riêng** nghĩa là khi lệnh kết thúc, toàn bộ RAM/VRAM của nó được hệ điều hành thu hồi — kernel notebook **không phình thêm chút nào**. Đây chính là lý do không gọi hàm render/metrics trực tiếp trong kernel sau khi RAM đã cao.

Đầu ra: `MODEL_PATH/test/ours_<iter>/{renders,gt}/*.png`, `results.json` (trung bình), `per_view.json` (từng ảnh).


In [ ]:
import os, json

results = None
if HAS_MODEL:
    !python render.py -m "{MODEL_PATH}" --skip_train --mult {MULT} --quiet
    !python metrics.py -m "{MODEL_PATH}"

    results_path = os.path.join(MODEL_PATH, "results.json")
    if os.path.exists(results_path):
        with open(results_path) as handle:
            results = json.load(handle)
        print(json.dumps(results, indent=2))
    else:
        print("metrics.py produced no results.json")
    show_mem("after render and metrics")
else:
    print("skipped: no trained model")


## 7.7 — Trình bày kết quả bằng matplotlib

Bốn khối hình (lưu kèm ra `MODEL_PATH/*.png`):

1. **Đường cong huấn luyện** — `Score` và 3 thành phần theo vòng lặp; PSNR để trục riêng vì đơn vị dB.
2. **$\Delta_{1k}$ dạng cột** — mức tăng/giảm `Score` mỗi 1000 vòng; xanh = tiến bộ, đỏ = thụt lùi.
3. **Số Gaussian & RAM/VRAM** theo vòng lặp — nhìn ra lúc nào densify làm phình điểm / căng bộ nhớ.
4. **Ảnh ví dụ**: GT — render — bản đồ lỗi $|I_{\text{render}}-I_{\text{gt}}|$ (heatmap), kèm PSNR từng ảnh.


In [ ]:
import os, pandas as pd, matplotlib.pyplot as plt

history_path = os.path.join(MODEL_PATH, "score_history.csv")
if not os.path.exists(history_path):
    print("skipped: no score_history.csv")
else:
    history = pd.read_csv(history_path)
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    ax = axes[0, 0]
    ax.plot(history["iter"], history["score"], "o-", lw=2.2, color="black", label="Score")
    ax.plot(history["iter"], 1 - history["lpips"], "s--", color="tab:red", label="1 - LPIPS")
    ax.plot(history["iter"], history["ssim"], "^--", color="tab:green", label="SSIM")
    ax.plot(history["iter"], history["psnr_norm"], "v--", color="tab:blue", label=r"$\widehat{PSNR}$")
    ax.set_xlabel("iteration")
    ax.set_ylabel("value in [0, 1]")
    ax.set_title("Score and components, measured every 1000 iterations")
    ax.legend(fontsize=8)
    ax.grid(alpha=.3)

    ax = axes[0, 1]
    delta = history["score"].diff()
    bar_width = max(float(history["iter"].diff().median() or 1000) * 0.7, 1.0)
    ax.bar(history["iter"], delta, width=bar_width,
           color=["tab:green" if value >= 0 else "tab:red" for value in delta.fillna(0)])
    ax.axhline(0, color="black", lw=.8)
    ax.set_xlabel("iteration")
    ax.set_ylabel(r"$\Delta_{1k}$ Score")
    ax.set_title("Score gain or loss per 1000 iterations")
    ax.grid(alpha=.3)

    ax = axes[1, 0]
    ax.plot(history["iter"], history["psnr"], "o-", color="tab:blue")
    ax.axhline(PSNR_MAX, ls="--", color="gray", lw=1, label=f"PSNR_max = {PSNR_MAX:.0f} dB")
    ax.set_xlabel("iteration")
    ax.set_ylabel("PSNR (dB)")
    ax.set_title("Absolute PSNR")
    ax.legend(fontsize=8)
    ax.grid(alpha=.3)

    ax = axes[1, 1]
    ax.plot(history["iter"], history["n_gauss"] / 1e6, "o-", color="tab:purple", label="gaussians (millions)")
    ax.set_xlabel("iteration")
    ax.set_ylabel("gaussians (millions)", color="tab:purple")
    twin = ax.twinx()
    twin.plot(history["iter"], history["ram_gb"], "s--", color="tab:orange", label="RAM (GB)")
    twin.plot(history["iter"], history["vram_gb"], "^--", color="tab:red", label="VRAM (GB)")
    twin.set_ylabel("GB")
    ax.set_title("Gaussian count and memory")
    lines = ax.get_lines() + twin.get_lines()
    ax.legend(lines, [line.get_label() for line in lines], fontsize=8, loc="lower right")
    ax.grid(alpha=.3)

    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_PATH, "training_curves.png"), dpi=110)
    plt.show()

    columns = [c for c in ["iter", "score", "d_score", "psnr", "ssim", "lpips",
                           "n_gauss", "ram_gb", "vram_gb", "elapsed_s"] if c in history.columns]
    display(history[columns].round(4))


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from PIL import Image

render_dir = os.path.join(MODEL_PATH, "test", f"ours_{ITERATIONS}", "renders")
gt_dir = os.path.join(MODEL_PATH, "test", f"ours_{ITERATIONS}", "gt")

if not (os.path.isdir(render_dir) and os.path.isdir(gt_dir)):
    print("skipped: no rendered test views")
else:
    names = sorted(os.listdir(render_dir))[:4]

    def image_psnr(a, b):
        mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
        return 99.0 if mse == 0 else 10 * np.log10(255.0 ** 2 / mse)

    fig, axes = plt.subplots(len(names), 3, figsize=(11, 3.1 * len(names)))
    if len(names) == 1:
        axes = axes[None, :]

    for row, name in enumerate(names):
        rendered = np.asarray(Image.open(os.path.join(render_dir, name)).convert("RGB"))
        truth = np.asarray(Image.open(os.path.join(gt_dir, name)).convert("RGB"))
        error = np.abs(rendered.astype(np.int16) - truth.astype(np.int16)).mean(-1)
        axes[row, 0].imshow(truth)
        axes[row, 0].set_ylabel(name, fontsize=8)
        axes[row, 1].imshow(rendered)
        axes[row, 1].set_title(f"render, PSNR {image_psnr(rendered, truth):.2f} dB")
        heatmap = axes[row, 2].imshow(error, cmap="inferno", vmin=0, vmax=40)
        if row == 0:
            axes[row, 0].set_title("ground truth")
            axes[row, 2].set_title(r"$|I_{render} - I_{gt}|$")
        for col in range(3):
            axes[row, col].set_xticks([])
            axes[row, col].set_yticks([])

    fig.colorbar(heatmap, ax=axes[:, 2], shrink=.6, label="mean absolute error per channel")
    plt.savefig(os.path.join(MODEL_PATH, "example_views.png"), dpi=110)
    plt.show()


## 7.8 — Thống kê kiểu Data Engineer

Gộp ba lớp hồ sơ thành một báo cáo có thể đính kèm:

| Bảng | Nội dung | Nguồn |
|---|---|---|
| **Hồ sơ dữ liệu** | số ảnh, phân giải, dung lượng, tách train/test, điểm COLMAP | 7.2 (`ds_summary`) |
| **Hồ sơ mô hình & huấn luyện** | số Gaussian, dung lượng `.ply`, bytes/Gaussian, thời gian, giây/1k vòng, đỉnh VRAM & RAM | `result` + `score_history.csv` |
| **Phân phối chất lượng theo view** | mean/std/min/median/max, tứ phân vị, 3 view tốt nhất & tệ nhất, tương quan PSNR–SSIM–LPIPS, histogram | `per_view.json` |

Xuất ra `stats.json`, `per_view_metrics.csv`, `REPORT.md`.


In [ ]:
import json, os, numpy as np, pandas as pd
import matplotlib.pyplot as plt

per_view_path = os.path.join(MODEL_PATH, "per_view.json")
if not (HAS_MODEL and results and os.path.exists(per_view_path)):
    print("skipped: metrics have not been computed")
else:
    history = pd.read_csv(os.path.join(MODEL_PATH, "score_history.csv"))
    ply_mb = os.path.getsize(PLY_PATH) / 1e6
    model_profile = dict(
        n_gauss=int(result["n_gauss"]),
        ply_MB=round(ply_mb, 1),
        bytes_per_gauss=round(ply_mb * 1e6 / max(result["n_gauss"], 1), 1),
        train_seconds=round(result["total_time"], 1),
        seconds_per_1k_iter=round(result["total_time"] / ITERATIONS * 1000, 2),
        peak_vram_GB=round(result["peak_vram_gb"], 2),
        peak_ram_GB=round(float(history["ram_gb"].max()), 2),
        holdout=result["holdout_kind"],
        final_live_score=round(float(history["score"].iloc[-1]), 4),
    )

    with open(per_view_path) as handle:
        per_view_raw = json.load(handle)
    method = list(per_view_raw.keys())[0]
    entries = per_view_raw[method]
    per_view = pd.DataFrame({
        "view": list(entries["PSNR"].keys()),
        "PSNR": list(entries["PSNR"].values()),
        "SSIM": list(entries["SSIM"].values()),
        "LPIPS": list(entries["LPIPS"].values()),
    })
    per_view["PSNR_norm"] = (per_view["PSNR"] / PSNR_MAX).clip(0, 1)
    per_view["Score"] = 0.4 * (1 - per_view["LPIPS"]) + 0.3 * per_view["SSIM"] + 0.3 * per_view["PSNR_norm"]
    per_view.to_csv(os.path.join(MODEL_PATH, "per_view_metrics.csv"), index=False)

    description = per_view[["PSNR", "SSIM", "LPIPS", "Score"]].describe(percentiles=[.25, .5, .75]).round(4)
    worst = per_view.nsmallest(3, "Score")[["view", "PSNR", "SSIM", "LPIPS", "Score"]]
    best = per_view.nlargest(3, "Score")[["view", "PSNR", "SSIM", "LPIPS", "Score"]]
    correlation = per_view[["PSNR", "SSIM", "LPIPS", "Score"]].corr().round(3)

    stats = dict(dataset=ds_summary, model=model_profile,
                 per_view_describe=json.loads(description.to_json()),
                 per_view_corr=json.loads(correlation.to_json()))
    with open(os.path.join(MODEL_PATH, "stats.json"), "w") as handle:
        json.dump(stats, handle, indent=2, default=str)

    print("dataset")
    print(json.dumps(ds_summary, indent=2, default=str))
    print("\nmodel and training")
    print(json.dumps(model_profile, indent=2))
    print("\nper view describe")
    print(description)
    print("\nworst three views")
    print(worst.to_string(index=False))
    print("\nbest three views")
    print(best.to_string(index=False))
    print("\ncorrelation")
    print(correlation)

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
    axes[0].hist(per_view["PSNR"], bins=15, color="tab:blue", edgecolor="white")
    axes[0].set_title("PSNR distribution across views")
    axes[0].set_xlabel("dB")
    axes[1].hist(per_view["Score"], bins=15, color="black", edgecolor="white")
    axes[1].set_title("Score distribution across views")
    scatter = axes[2].scatter(per_view["LPIPS"], per_view["PSNR"], c=per_view["Score"], cmap="viridis")
    axes[2].set_xlabel("LPIPS")
    axes[2].set_ylabel("PSNR (dB)")
    axes[2].set_title("PSNR versus LPIPS")
    fig.colorbar(scatter, ax=axes[2], label="Score")
    for ax in axes:
        ax.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_PATH, "per_view_stats.png"), dpi=110)
    plt.show()

    final_score = 0.4 * (1 - results[method]["LPIPS"]) + 0.3 * results[method]["SSIM"] \
        + 0.3 * min(results[method]["PSNR"] / PSNR_MAX, 1.0)

    with open(os.path.join(MODEL_PATH, "REPORT.md"), "w", encoding="utf-8") as handle:
        handle.write("# FastGS training report\n\n")
        handle.write(f"- source: `{SOURCE_PATH}`, holdout: {result['holdout_kind']}\n")
        handle.write(f"- iterations: {ITERATIONS}, time: {model_profile['train_seconds']}s "
                     f"({model_profile['seconds_per_1k_iter']}s per 1k)\n")
        handle.write(f"- gaussians: {model_profile['n_gauss']:,}, ply: {model_profile['ply_MB']} MB\n")
        handle.write(f"- final Score: **{final_score:.4f}** "
                     f"(PSNR {results[method]['PSNR']:.2f}, SSIM {results[method]['SSIM']:.4f}, "
                     f"LPIPS {results[method]['LPIPS']:.4f})\n")
        handle.write(f"- peak VRAM {model_profile['peak_vram_GB']} GB, "
                     f"peak RAM {model_profile['peak_ram_GB']} GB\n\n")
        handle.write("## per view describe\n\n```\n" + description.to_string() + "\n```\n")

    print(f"\nfinal Score from metrics.py: {final_score:.4f}")
    print("wrote REPORT.md, stats.json, per_view_metrics.csv to", MODEL_PATH)


## 7.9 — Tự động tải mô hình về máy (an toàn RAM)

Gói mô hình **ra đĩa** bằng `zipfile` với `ZIP_STORED` (không nén → gần như không tốn RAM; `.ply` vốn khó nén thêm). Chỉ đóng gói thứ cần để render lại (`point_cloud.ply`, `cfg_args`, `cameras.json`) cùng toàn bộ báo cáo và biểu đồ.

Hai đường lấy mô hình, điều khiển ở ô cấu hình 7.0:

| Công tắc | Mặc định | Hành vi |
|---|---|---|
| `DOWNLOAD_MODEL_TO_BROWSER` | `True` | gọi `files.download` → trình duyệt tải `.zip` về máy |
| `SAVE_MODEL_TO_DRIVE` | `False` | mount Drive rồi chép `.zip` vào `MyDrive`. **Tự bật khi file ≥ 200 MB** |

Cả hai đều bọc `try/except`: hỏng thì in đường dẫn `.zip` trên đĩa để bạn tự tải từ bảng Files bên trái, chứ không làm đứt Run all.

> Không dùng `io.BytesIO` hay `shutil.make_archive` gộp-vào-RAM — đó chính là thao tác làm OOM khi RAM đã cao sau train.


In [ ]:
import os, zipfile, gc, torch

if not HAS_MODEL:
    print("skipped: no trained model, nothing to package")
else:
    gc.collect()
    torch.cuda.empty_cache()
    show_mem("before packaging")

    ZIP_PATH = f"/content/fastgs_model_{ITERATIONS}.zip"
    INCLUDE = [
        f"point_cloud/iteration_{ITERATIONS}/point_cloud.ply",
        "cfg_args", "cameras.json", "input.ply",
        "score_history.csv", "stats.json", "REPORT.md",
        "per_view_metrics.csv", "results.json", "per_view.json",
        "training_curves.png", "example_views.png", "per_view_stats.png",
    ]

    with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_STORED) as archive:
        for relative in INCLUDE:
            path = os.path.join(MODEL_PATH, relative)
            if os.path.exists(path):
                archive.write(path, arcname=relative)
            else:
                print("skipping missing file:", relative)

    size_mb = os.path.getsize(ZIP_PATH) / 1e6
    print(f"archive ready: {ZIP_PATH} ({size_mb:.1f} MB)")
    show_mem("after packaging")

    if SAVE_MODEL_TO_DRIVE or size_mb >= 200:
        try:
            from google.colab import drive
            import shutil
            drive.mount("/content/drive")
            destination = "/content/drive/MyDrive/" + os.path.basename(ZIP_PATH)
            shutil.copy(ZIP_PATH, destination)
            print("copied to Drive:", destination)
        except Exception as exc:
            print("Drive copy failed:", exc)

    if DOWNLOAD_MODEL_TO_BROWSER:
        try:
            from google.colab import files
            print("starting browser download, allow multiple downloads if the browser asks")
            files.download(ZIP_PATH)
        except Exception as exc:
            print("browser download failed:", exc)
            print("the archive is still on disk at", ZIP_PATH)
            print("download it from the Files panel on the left, or set SAVE_MODEL_TO_DRIVE = True")
    else:
        print("browser download is off, set DOWNLOAD_MODEL_TO_BROWSER = True to enable it")
        print("the archive is on disk at", ZIP_PATH)


---

### Checklist chống tràn RAM trên Colab free T4

| Khi nào | Việc làm | Ô |
|---|---|---|
| Trong lúc train | `scene.save` ngay khi tới `save_iterations` → `point_cloud.ply` an toàn trên đĩa **trước khi** RAM căng | 7.4 |
| Mỗi 1000 vòng | `torch.cuda.empty_cache()`; nếu RAM > `RAM_SOFT_LIMIT_GB` ⇒ `gc.collect()` ngay | 7.4 |
| Mỗi vòng lặp | `del` mọi tensor trung gian | 7.4 |
| Ngay sau train | `del gaussians, scene`; `gc.collect()`; `empty_cache()`; `ipc_collect()` | 7.5 |
| Render + metrics | chạy bằng `!python …` (**tiến trình con**) — RAM/VRAM được HĐH thu hồi khi lệnh thoát | 7.6 |
| Đóng gói | `zipfile` chế độ `ZIP_STORED` ghi **ra đĩa**, tuyệt đối không `BytesIO` / `shutil.make_archive` vào RAM | 7.9 |
| File > ~200 MB | chép sang Google Drive thay cho `files.download` | 7.9 |

Nhờ vậy kernel **không bao giờ** giữ đồng thời "mô hình trong RAM + bản nén trong RAM" → train xong vẫn tải được mô hình về máy.

### Đọc thêm

| Tài liệu | Nội dung |
|---|---|
| [DOCS/colab-t4-guide.md](DOCS/colab-t4-guide.md) | Bản văn xuôi của Phần 7–9: preset A và lý do từng tham số, biên lợi ích của `Score`, playbook RAM, ba nâng cấp, roadmap CUDA |
| [DOCS/fastgs-acceleration-method.md](DOCS/fastgs-acceleration-method.md) | Cơ chế FastGS (Phần 1–6) + ba điều mã nguồn nói mà README không nói |
| [DOCS/gaussian-splatting-math.md](DOCS/gaussian-splatting-math.md) | Nền tảng toán học 3DGS |
| [DOCS/README.md](DOCS/README.md) | Mục lục tài liệu + bản đồ tài liệu ↔ mã nguồn |

Mã thật notebook này bám theo: `train.py`, `render.py`, `metrics.py`, `utils/fast_utils.py`, `scene/gaussian_model.py`, `scene/__init__.py`, `utils/camera_utils.py`.

---

# Phần 8: Ba nâng cấp thuần Python + ablation

Phần 7 là baseline **preset A** (tham số đã được kiểm chứng trong `train_base.sh`). Phần này bật ba nâng cấp đã cài sẵn trong `train_fastgs` và **đo xem chúng có thật sự giúp không** — thay vì tin lời hứa trên giấy.

### 8.1 — Coarse-to-fine (ý tưởng DashGaussian)

Dùng sẵn `Scene(resolution_scales=[...])` của repo: nạp camera ở nhiều độ phân giải, rồi đổi theo lịch

$$r(t)=\begin{cases}
4.0 & t < 0.10\,T\\
2.0 & 0.10\,T \le t < 0.27\,T\\
1.0 & t \ge 0.27\,T
\end{cases}$$

Với `-r 2`, scale 4.0 nghĩa là ảnh bằng **1/8** cạnh gốc ⇒ chi phí mỗi vòng còn ~1/64 số pixel. Vòng lặp đắt nhất (0–15k, có densify + Adam đầy đủ) được rút gọn mạnh, và chuyển về full-res tại 0.27·T vẫn còn ~7k vòng densify ở độ phân giải thật.

Hai chi tiết đúng đắn đã xử lý:
- `max_screen_size = 20 px` **chỉ áp dụng khi `scale == 1.0`** — ngưỡng theo pixel vô nghĩa ở ảnh nhỏ;
- `max_radii2D` được **reset** mỗi lần đổi scale để không trộn bán kính giữa hai độ phân giải;
- hold-out để tính Score **luôn ở scale 1.0** nên đường cong Score so sánh được xuyên suốt.

> Đánh đổi: nạp 3 bộ camera ⇒ tốn thêm bộ nhớ ảnh GT (≈ +30% so với chỉ scale 1.0). Nếu VRAM căng, dùng `resolution_scales=(2.0, 1.0)`.

### 8.2 — Ngân sách primitive (ý tưởng Taming-3DGS / 3DGS-MCMC)

FastGS chặn tăng trưởng bằng ngưỡng cứng `importance_score > 5`. Ta thêm một **trần tuyệt đối** có lịch:

$$B(t) = N_0 + (N_{\max}-N_0)\Big(\tfrac{t}{0.5\,T}\Big)^{p}$$

Sau mỗi lần densify, nếu $n > B(t)$ thì cắt $n - B(t)$ Gaussian có **opacity thấp nhất** (`enforce_budget`). Số điểm bị chặn trần ⇒ thời gian mỗi vòng và VRAM đều bị chặn ⇒ **không bao giờ OOM giữa chừng trên T4**.

### 8.3 — SGLD noise + phạt L1 (ý tưởng 3DGS-MCMC)

$$\mathbf{x}_i \leftarrow \mathbf{x}_i + \underbrace{\sigma\big(-k(o_i-\tau)\big)}_{\text{cổng: chỉ đá Gaussian mờ}} \cdot\ \mathbf{s}_i \odot \boldsymbol{\epsilon}\ \cdot \lambda\,\eta_t,
\qquad \boldsymbol{\epsilon}\sim\mathcal{N}(0, I)$$

cộng hai số hạng phạt vào loss: $\lambda_o\,\overline{o} + \lambda_s\,\overline{s}$. Nhiễu tỉ lệ với **scale riêng** của từng Gaussian và lr vị trí hiện tại, chỉ bật khi `iteration < densify_until_iter` (giai đoạn optimizer còn bước mỗi vòng).

---

### Ablation

Ô dưới chạy **3 cấu hình × `ABLATION_ITERS` vòng**, cùng dataset, cùng preset A, chỉ khác các cờ. Xuất `ablation.csv` + biểu đồ **Score theo phút** — đây mới là đường cong đáng đưa vào hồ sơ, vì nó trả lời trực tiếp "chất lượng trên mỗi phút GPU".

| Cấu hình | Bật gì |
|---|---|
| `presetA` | không bật gì (đối chứng) |
| `coarse2fine` | 8.1 |
| `c2f_budget_mcmc` | 8.1 + 8.2 + 8.3 |

> ⏱ Mặc định `ABLATION_ITERS = 7000` cho 3 lần chạy. Trên T4 free ước tính ~30–60 phút tổng tuỳ scene. Giảm xuống 4000 nếu phiên Colab ngắn. Giữa các lần chạy đã có `del` + `gc.collect()` + `empty_cache()` nên RAM không cộng dồn.


In [ ]:
import os, gc, json, torch, pandas as pd
import matplotlib.pyplot as plt

ABLATION_ITERS = 4000
ABLATION_ROOT = "/content/output/ablation"

CONFIGS = [
    dict(tag="presetA",
         resolution_scales=(1.0,), resolution_switch_fracs=(),
         gaussian_budget=None, mcmc_noise=0.0, lambda_opacity=0.0, lambda_scale=0.0),
    dict(tag="coarse2fine",
         resolution_scales=(4.0, 2.0, 1.0), resolution_switch_fracs=(0.10, 0.27),
         gaussian_budget=None, mcmc_noise=0.0, lambda_opacity=0.0, lambda_scale=0.0),
    dict(tag="c2f_budget_mcmc",
         resolution_scales=(4.0, 2.0, 1.0), resolution_switch_fracs=(0.10, 0.27),
         gaussian_budget=1_200_000, mcmc_noise=2.0, lambda_opacity=0.001, lambda_scale=0.001),
]


def run_ablation(config, iterations=ABLATION_ITERS):
    tag = config["tag"]
    model_path = os.path.join(ABLATION_ROOT, tag)
    cfg_args, lp, op, pp = build_args(model_path, preset_a_args(iterations))
    options = {key: value for key, value in config.items() if key != "tag"}

    summary, gaussians, scene = train_fastgs(
        lp.extract(cfg_args), op.extract(cfg_args), pp.extract(cfg_args),
        score_every=SCORE_EVERY, eval_views=EVAL_VIEWS,
        save_iterations=(iterations,), psnr_max=PSNR_MAX,
        tag=tag, **options)

    curve = pd.DataFrame(summary["history"])
    curve.to_csv(os.path.join(model_path, "score_history.csv"), index=False)
    last = curve.iloc[-1]
    row = dict(config=tag,
               score=float(last["score"]),
               psnr=float(last["psnr"]),
               ssim=float(last["ssim"]),
               lpips=float(last["lpips"]),
               n_gauss=int(summary["n_gauss"]),
               minutes=summary["total_time"] / 60.0,
               peak_vram_gb=summary["peak_vram_gb"])

    del gaussians, scene, summary
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    show_mem(f"after {tag}")
    return row, curve


if not (RUN_ABLATION and DATA_READY):
    print(f"skipped: RUN_ABLATION={RUN_ABLATION}, DATA_READY={DATA_READY}")
    print("set RUN_ABLATION = True in the config cell to run the three comparison trainings")
else:
    rows, curves = [], {}
    for config in CONFIGS:
        row, curve = run_ablation(config)
        rows.append(row)
        curves[row["config"]] = curve
        print(json.dumps(row, indent=2, default=str))

    ablation = pd.DataFrame(rows)
    ablation["score_per_minute"] = ablation["score"] / ablation["minutes"]
    os.makedirs(ABLATION_ROOT, exist_ok=True)
    ablation.to_csv(os.path.join(ABLATION_ROOT, "ablation.csv"), index=False)

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for name, curve in curves.items():
        axes[0].plot(curve["iter"], curve["score"], "o-", label=name)
        axes[1].plot(curve["elapsed_s"] / 60.0, curve["score"], "o-", label=name)
    axes[0].set_xlabel("iteration")
    axes[0].set_ylabel("Score")
    axes[0].set_title("Score versus iteration")
    axes[1].set_xlabel("minutes")
    axes[1].set_ylabel("Score")
    axes[1].set_title("Score versus wall clock")
    for ax in axes[:2]:
        ax.legend(fontsize=8)
        ax.grid(alpha=.3)
    axes[2].bar(ablation["config"], ablation["minutes"], color="tab:orange")
    axes[2].set_ylabel("minutes")
    axes[2].set_title("Training time")
    axes[2].tick_params(axis="x", rotation=20)
    axes[2].grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(os.path.join(ABLATION_ROOT, "ablation.png"), dpi=110)
    plt.show()

    display(ablation.round(4))


---

# Phần 9: Những gì tôi **cố tình chưa** wire (cần đụng CUDA)

Ba nâng cấp ở Phần 8 đều là **thuần Python** nên kiểm chứng được ngay trong notebook. Các nâng cấp dưới đây mạnh hơn nhưng đòi hỏi sửa/thay kernel CUDA — tôi ghi rõ trạng thái thay vì ship code không chạy thử được.

### 9.1 — Mip‑Splatting: cờ `--antialiasing` trong repo này là **cờ chết**

Đã kiểm tra: `gaussian_renderer/__init__.py` dựng `GaussianRasterizationSettings(...)` **không có** trường `antialiasing`, và `submodules/diff-gaussian-rasterization_fastgs/` không tham chiếu chữ `antialiasing` ở đâu. Truyền cờ này chỉ bị bỏ qua âm thầm.

Muốn có thật, phải sửa trong kernel công thức alpha mỗi fragment:

$$\alpha_i = o_i\;\underbrace{\sqrt{\frac{|\Sigma'_i|}{|\Sigma'_i + s\,I|}}}_{\text{3DGS thiếu hệ số này}}\;
\exp\!\Big(-\tfrac12\,\Delta^\top(\Sigma'_i + s\,I)^{-1}\Delta\Big),\qquad s \approx 0.1\ \text{px}^2$$

thay cho phép giãn cố định $\Sigma' \leftarrow \Sigma' + 0.3\,I$. Kèm bộ lọc 3D kẹp phương sai tối thiểu theo tần số lấy mẫu lớn nhất $\hat s = \max_k f_k/d_k$, tức $\Sigma \leftarrow \Sigma + \hat s^{-2} I$.

Các bước: sửa `forward.cu` / `backward.cu` trong submodule → thêm trường vào `GaussianRasterizationSettings` → truyền `pipe.antialiasing` từ `render_fastgs` → `pip install ./submodules/...` lại. **Ưu tiên số 1** cho ảnh drone vì tỉ lệ biến thiên lớn trong cùng khung hình.

### 9.2 — StopThePop: sắp xếp đúng ở mức pixel

Khoá sắp xếp hiện tại là độ sâu **tâm** Gaussian trên mỗi tile 16×16. Đúng phải là độ sâu tại điểm đóng góp cực đại dọc tia pixel:

$$t^\* = \arg\max_t\ \mathcal{G}\big(\mathbf{o} + t\,\mathbf{d}\big), \qquad \text{key} = z(t^\*)$$

Cách làm: thay submodule rasterizer bằng kernel StopThePop, giữ nguyên phần Python. Hết popping, tốc độ tương đương hoặc nhỉnh hơn nhờ cull tốt hơn.

### 9.3 — 3DGS‑MCMC đầy đủ

Phần 8 đã có **nhiễu SGLD có cổng** và **phạt L1**, nhưng còn thiếu mảnh quan trọng nhất: **di dời bảo toàn ảnh render**. Khi chuyển $N$ bản sao về một vị trí, opacity mới phải thoả

$$1-(1-o_{\text{new}})^{N} = o_{\text{old}} \;\Longrightarrow\; o_{\text{new}} = 1-(1-o_{\text{old}})^{1/N}$$

kèm hiệu chỉnh $\Sigma_{\text{new}}$ khớp mô‑men bậc hai. Cần viết lại luồng densify, nên đường ngắn hơn là **chuyển backend sang `gsplat`** (đã có sẵn strategy MCMC + antialiasing), rồi port lại điểm số đa góc nhìn của FastGS lên đó.

### 9.4 — 2DGS / GOF cho digital twin

3DGS và FastGS tối ưu cho **ảnh mới**, không cho **hình học**. Nếu mục tiêu là mesh/bề mặt, primitive đúng là đĩa phẳng với giao tia chính xác (không cần Jacobian $J$):

$$\mathbf{x}(u,v)=\mathbf{p}_c + s_u u\,\mathbf{t}_u + s_v v\,\mathbf{t}_v,
\qquad \mathcal{G}(u,v)=\exp\!\big(-\tfrac{u^2+v^2}{2}\big)$$

Đổi sang `diff-surfel-rasterization` + thêm hai regularizer (depth‑distortion, normal‑consistency). Đây là **dự án riêng**, không phải một cờ.

### 9.5 — Khởi tạo MASt3R/DUSt3R

Repo đã tham chiếu submodule `dust3r`. Thay điểm COLMAP thưa bằng pointmap dày $X \in \mathbb{R}^{H\times W\times 3}$ giúp rất nhiều cho ảnh trên không / ít chồng lấp. Việc cần làm: viết `sceneLoadTypeCallbacks["Dust3r"]` trả về `BasicPointCloud` từ pointmap, còn lại giữ nguyên.

---

### Thứ tự tôi khuyến nghị nếu tiếp tục

1. Chạy Phần 8 → có bảng ablation thật với số liệu của **máy bạn**.
2. Vá Mip‑Splatting vào kernel (9.1) — lợi/chi phí tốt nhất.
3. Nếu là digital twin cần mesh → nhánh 2DGS/GOF (9.4).
4. Nếu cần chất lượng trên mỗi primitive → gsplat + MCMC (9.3).
